# Embeddings & evaluation

A focused companion to [`walkthrough.ipynb`](walkthrough.ipynb). The walkthrough runs the whole
pipeline end to end; this one zooms in on two ideas:

1. **What an embedding is** — text turned into a vector so similar meanings sit near each other.
2. **How to tell whether retrieval is any good** — where eyeballing stops working, and how a
   gold set plus `hit@k` / `MRR` let you compare embedders with numbers.

It runs **key-free** — no API key of any kind: everything uses small local models, and nothing
here calls generation, so nothing is billed.

Launch from the repo root: `uv run jupyter lab`.

In [ ]:
import os
from pathlib import Path

from IPython.display import Markdown, display

# Resolve the repo root so ./data/index and ./eval/gold work whether Jupyter was
# launched at the root or inside notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)

# No key needed: this notebook only embeds, retrieves, and evaluates — all on local
# models. We still load .env so a key is picked up if you've set one (e.g. to try
# the Voyage comparison in section 7).
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")


def show_table(headers, rows):
    """Render rows as a real table rather than tab-aligned text."""
    def cell(v):
        return str(v).replace("|", "\\|").replace("<", "&lt;")
    lines = ["| " + " | ".join(headers) + " |", "|" + " --- |" * len(headers)]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


print("ready — repo root:", ROOT.name)

## 1. What an embedding is

An embedding turns text into a **vector** — a fixed-length list of numbers, the same length for
every text, short or long — chosen so that **similar meanings land near each other**. No single
number means anything on its own; what carries meaning is the *direction* the whole list points,
so the *same* model must embed both the stored rules and your question or the geometry wouldn't
line up. Below, embed **two pairs of near-paraphrases** — two ways of saying one checkers idea,
two ways of saying one cornhole idea — print one vector (truncated) to see what they actually
are, then look at how close they land (cosine similarity; 1.0 = same direction).

In [ ]:
import numpy as np
from rulebook.embeddings import get_embedder

emb = get_embedder()  # the default local model — no key, no call to a hosted provider
phrases = [
    "winning a game of checkers",                 # p0 ─┐ same idea,
    "how you win at checkers",                    # p1 ─┘ said two ways
    "tossing a beanbag onto the cornhole board",  # p2 ─┐ same idea,
    "throwing a bag in cornhole",                 # p3 ─┘ said two ways
]
vecs = np.asarray(emb.embed(phrases, input_type="document"))

print(f"each embedding is a fixed-length vector of {vecs.shape[1]} numbers")
print(f"p0, first 8 of {vecs.shape[1]}:  {np.round(vecs[0][:8], 3)} ...")
print(f"its length (norm) is {np.linalg.norm(vecs[0]):.2f}  "
      f"(unit vectors, so the dot products below ARE cosine similarities)")

sims = vecs @ vecs.T
labels = [f"p{i}" for i in range(len(phrases))]
show_table(
    [""] + labels + ["phrase"],
    [[lab] + [f"{v:.2f}" for v in row] + [p] for lab, row, p in zip(labels, sims, phrases)],
)

Each phrase is nearest its paraphrase partner — `p0↔p1` and `p2↔p3` are the biggest
off-diagonal numbers, and similarity drops across the two topics: *meaning*, not matching words,
decides nearness. It's a blunt signal, though — this is a lightweight static model — and you'll
see it bite in §4.

## 2. A tiny index to search

Retrieval needs an index: every rule chunked and embedded. Build the bundled checkers sample if
it isn't already — this is exactly what the quickstart's `build_index.py` does, key-free.

In [ ]:
import subprocess, sys
from rulebook.config import settings
from rulebook.store import list_domains

if "checkers" in list_domains(settings.resolved_index_path):
    print("checkers: already built")
else:
    print("checkers: building...")
    subprocess.run([sys.executable, "scripts/build_index.py", "--domain", "checkers"], check=True)

## 3. Retrieve: an obvious winner

Given a question, embed it the same way and pull the nearest rules. **Distance** is what
retrieval sorts on — lower is nearer (it's the distance between two unit vectors, so 0 means
identical and 2 means opposite).

In [ ]:
from rulebook.retrieve import retrieve


def show_top(question, k=3):
    """The top-k rules for a question, as a table."""
    rows = [
        (rank, c.rule_id, f"{c.distance:.2f}", c.text[:80] + "…")
        for rank, c in enumerate(retrieve(question, domain="checkers", k=k), start=1)
    ]
    show_table(["rank", "rule", "distance", "text"], rows)


show_top("If I touch one of my own pieces, must I move it?")

The touch-and-move rule comes first, well ahead of the runner-up — and you can tell it's the
right answer just by reading it. For a question like this, eyeballing works.

## 4. Retrieve: a muddled one

Now a question whose answer is just as plainly in the rules — *Object of the game* and *Winning*:

In [ ]:
show_top("How do you win a game of checkers?")

Look at what came first: the **preamble** — the file's license note and introduction, not a rule
at all. And it won by a wide margin, so the model was *confidently* wrong. The rules that actually
answer the question are further down. Hand only the top result to an LLM and it gets the wrong
context.

(Cleaning boilerplate like that out of documents before indexing would be another fix — a
chunking change. Here, let's try changing the model instead.)

## 5. Change one thing: the embedder

An **embedder** is the model that turns text into the vector from §1. Swap it and you change the
geometry — so the rankings change.

The three local embedders below are **model2vec "potion"** models from
[MinishLab](https://huggingface.co/minishlab) on Hugging Face: small *static* models distilled
from larger embedding models, downloaded once and run on your CPU. They differ mainly in size
(2M / 8M / 32M parameters), and one is tuned specifically for retrieval. The app also supports two
hosted embedders, **Voyage** and **OpenAI**, and many other providers exist.

| | Local (model2vec) | Hosted (Voyage, OpenAI) |
|---|---|---|
| Cost | free | per token (small) |
| Setup | none | API key + account |
| Privacy | text never leaves your machine | text is sent to the provider |
| Quality | small, static (ignores word context) | larger, contextual |
| Offline | yes | no |

Below, the same two questions go through all three. Each model gets its own temporary index, so
your real index isn't touched. (The first run downloads two more small models.)

In [ ]:
from rulebook.eval import build_scratch_index, rank_of

MODELS = [
    "minishlab/potion-retrieval-32M",
    "minishlab/potion-base-8M",
    "minishlab/potion-base-2M",
]
indexes = {m: build_scratch_index("checkers", m) for m in MODELS}


def label(model):
    name = model.split("/")[-1]
    return f"{name} (default)" if model == settings.embedding_model else name


def where(question, gold, index):
    r = rank_of(question, gold, index=index)
    return "not in top 5" if r is None else ("**#1**" if r == 1 else f"#{r}")


QUESTIONS = [
    ("If I touch one of my own pieces, must I move it?", ("5.3",)),
    ("How do you win a game of checkers?", ("5.1", "2.1")),
]
show_table(
    ["question", "right rule"] + [label(m) for m in MODELS],
    [[q, " / ".join(g)] + [where(q, g, indexes[m]) for m in MODELS] for q, g in QUESTIONS],
)

The first question was never in doubt. The second one depends entirely on the embedder — compare
where the answer lands in each column.

One thing that *didn't* change: **`k`**, how many passages retrieval returns. `k` never reorders
results; it only decides how many the LLM gets to read. If a model puts the answer at #3, then
`k=1` hands the LLM the wrong passage and `k=3` only just includes the right one.

So should you switch models — on the strength of **two** questions? You'd want to check many
more. And the next time you change something (the model, the chunking, `k`), you'd want to check
them all again, the same way.

## 6. Write the answers down once: a gold set

When you looked at §3 and saw that touch-and-move was right, you were checking against an answer
you already knew. A **gold set** is that knowledge written down: a list of questions, each paired
with the rule (or rules) that answer it. Write it once, and you can score *any* retrieval setup
against it — automatically, and the same way every time.

Here's a minimal gold set for checkers (it lives in `eval/gold/checkers.jsonl`):

In [ ]:
from rulebook.eval import load_gold

gold = load_gold("checkers")
show_table(["question", "right rule(s)"], [(g.question, " / ".join(g.gold)) for g in gold])

Two standard scores turn "did retrieval find it?" into numbers:

- **hit@k** — the share of questions whose right rule shows up in the top *k*. *Is the answer
  even in what the LLM gets to read?*
- **MRR** (mean reciprocal rank) — score each question 1 / (rank of the right rule): #1 → 1.0,
  #2 → 0.5, #3 → 0.33, missing → 0; then average. It rewards putting the answer *first*, not just
  somewhere.

The same `evaluate()` runs here and behind `python -m rulebook.eval`, so the notebook and the app
report the same numbers.

In [ ]:
from rulebook.eval import evaluate

rows = []
for m in MODELS:
    res = evaluate(gold, k=5, index=indexes[m])
    rows.append([label(m)] + [f"{res.hit_rate_at(k):.2f}" for k in (1, 3, 5)] + [f"{res.mrr:.2f}"])
show_table(["model", "hit@1", "hit@3", "hit@5", "MRR"], rows)

What two questions couldn't tell you:

- **"Better" isn't one number.** A model can put the right rule *first* more often (higher hit@1)
  and still miss questions outright (lower hit@5). Which matters more depends on `k`: this app
  hands the LLM 5 passages, so a rule at #4 is fine and a rule that's missing entirely is not.
- **Bigger, or "tuned for retrieval," isn't automatically better.** Check where the
  retrieval-tuned model lands before assuming it wins.
- **Mind the sample size.** With 12 questions, one question moves hit@k by about 0.08. A gold set
  doesn't have to be big to be useful, but it's evidence, not proof.

## 7. Local vs. a hosted embedder (Voyage)

The same gold set answers *"is a hosted model worth it?"* with a number. Hosted models aren't
local, so this one uses the real index (needs a `VOYAGE_API_KEY`, which costs a little):

```bash
# 1. rebuild the checkers index with Voyage vectors
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python scripts/build_index.py --domain checkers
# 2. score it against the same gold set
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python -m rulebook.eval --domain checkers
# 3. put the default back when you're done
uv run python scripts/build_index.py --domain checkers
```

Compare its scores with the table above. On a small, clean corpus like this, a hosted model may
not win by much — and that's a useful result too. Either way it's evidence instead of a guess.

## Takeaway

- An embedding is a fixed-length vector; retrieval finds the vectors nearest the question.
- One question at a time, you can judge results by eye — until the answer is buried, or you want
  to compare two setups.
- A **gold set** is your judgment written down once. **hit@k** and **MRR** score any setup
  against it, so "is this change better?" gets a number.
- Measure before you upgrade: the bigger or fancier model isn't always the better one.